### Kernel: Python

## Rede C com Optuna (tempo continuo)

In [12]:
# Importações

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import optuna

SEED = 2003
np.random.seed(SEED)
torch.manual_seed(SEED)

In [13]:
entrada_rede = pd.read_csv("../dados/entrada/entrada_rede.csv")

dados_treino = entrada_rede.loc[entrada_rede["conjunto"] == "treino"].copy()
dados_teste = entrada_rede.loc[entrada_rede["conjunto"] == "teste"].copy()

tempos_avaliacao_teste = np.arange(1, 132, 1)

In [14]:
def normaliza_coluna_numerica(serie, stats_coluna):
    valores = pd.to_numeric(serie, errors="coerce").fillna(stats_coluna["mediana"])
    return (valores - stats_coluna["media"]) / (stats_coluna["desvio"] + 1e-8)

covariaveis = [
    col for col in dados_treino.columns
    if col not in ["id_paciente", "tempo", "pseudo", "conjunto"]
]
covariaveis_numericas = [
    col for col in covariaveis if pd.api.types.is_numeric_dtype(dados_treino[col])
]
covariaveis_categoricas = [
    col for col in covariaveis if col not in covariaveis_numericas
]

estatisticas_numericas = {}
for col in covariaveis_numericas:
    serie = pd.to_numeric(dados_treino[col], errors="coerce")
    mediana = float(serie.median()) if serie.notna().any() else 0.0
    serie_imp = serie.fillna(mediana)
    estatisticas_numericas[col] = {
        "mediana": mediana,
        "media": float(serie_imp.mean()),
        "desvio": float(serie_imp.std(ddof=0)),
    }

tempo_treino = pd.to_numeric(dados_treino["tempo"], errors="coerce").astype(float)
stats_tempo = {
    "mediana": float(tempo_treino.median()),
    "media": float(tempo_treino.mean()),
    "desvio": float(tempo_treino.std(ddof=0)),
}

mapeamento_categorias = {}
for col in covariaveis_categoricas:
    mapeamento_categorias[col] = sorted(
        dados_treino[col].fillna("NA").astype(str).unique().tolist()
    )

def gera_matriz_design_tempo_continuo(dados):
    dados_ref = dados.reset_index(drop=True).copy()

    base_num = pd.DataFrame(index=dados_ref.index)

    for col in covariaveis_numericas:
        base_num[col] = normaliza_coluna_numerica(
            dados_ref[col],
            estatisticas_numericas[col]
        )

    base_num["tempo"] = normaliza_coluna_numerica(
        dados_ref["tempo"],
        stats_tempo
    )

    if len(covariaveis_categoricas) > 0:
        base_cat = dados_ref[covariaveis_categoricas].copy()

        for col in covariaveis_categoricas:
            base_cat[col] = pd.Categorical(
                base_cat[col].fillna("NA").astype(str),
                categories=mapeamento_categorias[col],
                ordered=False,
            )

        base_cat = pd.get_dummies(
            base_cat,
            prefix=covariaveis_categoricas,
            dtype=float
        ).reset_index(drop=True)
    else:
        base_cat = pd.DataFrame(index=dados_ref.index)

    return pd.concat([base_num, base_cat], axis=1)

In [15]:
rng = np.random.default_rng(SEED)
indices = np.arange(len(dados_treino))
rng.shuffle(indices)

# Divisao em treino e validacao dos dados de treino originais
split = int(0.8 * len(indices))
idx_train = indices[:split]
idx_val = indices[split:]

dados_treino_opt = dados_treino.iloc[idx_train].copy()
dados_val_opt = dados_treino.iloc[idx_val].copy()

x_train_opt = gera_matriz_design_tempo_continuo(dados_treino_opt)
y_train_opt = dados_treino_opt["pseudo"].astype(float).values.reshape(-1, 1)
x_val_opt = gera_matriz_design_tempo_continuo(dados_val_opt)
y_val_opt = dados_val_opt["pseudo"].astype(float).values.reshape(-1, 1)

train_ds = TensorDataset(
    torch.tensor(x_train_opt.values.astype(np.float32)),
    torch.tensor(y_train_opt.astype(np.float32)),
)
val_ds = TensorDataset(
    torch.tensor(x_val_opt.values.astype(np.float32)),
    torch.tensor(y_val_opt.astype(np.float32)),
)

In [16]:
# Definição da arquitetura da rede
class rede_optuna(nn.Module):
    def __init__(self, input_dim, layers):
        super().__init__()
        modules = []
        prev = input_dim
        for h in layers:
            modules.append(nn.Linear(prev, h))
            modules.append(nn.Tanh())
            prev = h
        modules.append(nn.Linear(prev, 1))
        modules.append(nn.Sigmoid())
        self.net = nn.Sequential(*modules)

    def forward(self, x):
        return self.net(x)

In [23]:
   
# Grade de busca de hiperparâmetros com Optuna

def objective(trial):
    n_layers = trial.suggest_int("n_layers", 1, 12)
    layers = [trial.suggest_int(f"n_units_{i}", 8, 128) for i in range(n_layers)]
    lr = trial.suggest_float("lr", 0.0001, 0.05, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
    weight_decay = trial.suggest_float("weight_decay", 1e-8, 1e-3, log=True)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = rede_optuna(input_dim=x_train_opt.shape[1], layers=layers)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(250):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_losses = []
            for xb, yb in val_loader:
                preds = model(xb)
                val_losses.append(criterion(preds, yb).item())
            val_loss = float(np.mean(val_losses))

        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    print(f"Trial {trial.number} | val_loss={val_loss:.6f} | params={trial.params}", flush=True)
    return val_loss

pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
study = optuna.create_study(direction="minimize", pruner=pruner)
study.optimize(objective, n_trials=20)

best_params = study.best_params
print("Melhores parametros:", best_params)
print("MSE validacao:", study.best_value)

[I 2026-05-12 20:13:02,436] A new study created in memory with name: no-name-2dd59892-7dfa-429a-bb7b-f789c46e60a8


Trial 0 | val_loss=0.037347 | params={'n_layers': 5, 'n_units_0': 95, 'n_units_1': 18, 'n_units_2': 21, 'n_units_3': 24, 'n_units_4': 44, 'lr': 0.0002106307432953969, 'batch_size': 64, 'weight_decay': 6.576957704948965e-05}


[I 2026-05-12 20:22:46,169] Trial 0 finished with value: 0.037346777885240476 and parameters: {'n_layers': 5, 'n_units_0': 95, 'n_units_1': 18, 'n_units_2': 21, 'n_units_3': 24, 'n_units_4': 44, 'lr': 0.0002106307432953969, 'batch_size': 64, 'weight_decay': 6.576957704948965e-05}. Best is trial 0 with value: 0.037346777885240476.


Trial 1 | val_loss=0.066379 | params={'n_layers': 10, 'n_units_0': 96, 'n_units_1': 111, 'n_units_2': 122, 'n_units_3': 71, 'n_units_4': 64, 'n_units_5': 13, 'n_units_6': 128, 'n_units_7': 21, 'n_units_8': 24, 'n_units_9': 68, 'lr': 0.0011822637808979245, 'batch_size': 128, 'weight_decay': 6.306598564132205e-08}


[I 2026-05-12 20:32:23,469] Trial 1 finished with value: 0.06637933727745947 and parameters: {'n_layers': 10, 'n_units_0': 96, 'n_units_1': 111, 'n_units_2': 122, 'n_units_3': 71, 'n_units_4': 64, 'n_units_5': 13, 'n_units_6': 128, 'n_units_7': 21, 'n_units_8': 24, 'n_units_9': 68, 'lr': 0.0011822637808979245, 'batch_size': 128, 'weight_decay': 6.306598564132205e-08}. Best is trial 0 with value: 0.037346777885240476.


...

Trial 15 | val_loss=0.029205 | params={'n_layers': 3, 'n_units_0': 128, 'n_units_1': 80, 'n_units_2': 107, 'lr': 0.0008400120414566779, 'batch_size': 64, 'weight_decay': 2.068814540107844e-06}


[I 2026-05-12 22:21:14,785] Trial 15 finished with value: 0.02920462885441865 and parameters: {'n_layers': 3, 'n_units_0': 128, 'n_units_1': 80, 'n_units_2': 107, 'lr': 0.0008400120414566779, 'batch_size': 64, 'weight_decay': 2.068814540107844e-06}. Best is trial 11 with value: 0.024090384268962033.
[I 2026-05-12 22:21:50,359] Trial 16 pruned. 
[I 2026-05-12 22:21:58,159] Trial 17 pruned. 
[I 2026-05-12 22:22:20,630] Trial 18 pruned. 


Trial 19 | val_loss=0.028233 | params={'n_layers': 4, 'n_units_0': 82, 'n_units_1': 62, 'n_units_2': 61, 'n_units_3': 56, 'lr': 0.0005444782218011857, 'batch_size': 32, 'weight_decay': 8.959217566149604e-08}


[I 2026-05-12 22:33:11,198] Trial 19 finished with value: 0.028233130850079953 and parameters: {'n_layers': 4, 'n_units_0': 82, 'n_units_1': 62, 'n_units_2': 61, 'n_units_3': 56, 'lr': 0.0005444782218011857, 'batch_size': 32, 'weight_decay': 8.959217566149604e-08}. Best is trial 11 with value: 0.024090384268962033.


Melhores parametros: {'n_layers': 2, 'n_units_0': 126, 'n_units_1': 85, 'lr': 0.0006334398912380633, 'batch_size': 64, 'weight_decay': 2.790293791588141e-07}
MSE validacao: 0.024090384268962033


In [17]:
best_params = {
        "n_layers": 2,
        "n_units_0": 126,
        "n_units_1": 85,
        "lr": 0.0006334398912380633,
        "batch_size": 64,
        "weight_decay": 2.790293791588141e-07,
    }
best_value = 0.024090384268962033

In [18]:
# Melhores hiperparametros encontrados e treinamento final com todos os dados de treino originais
layers_opt = [best_params[f"n_units_{i}"] for i in range(best_params["n_layers"])]
batch_size_opt = best_params["batch_size"]
lr_opt = best_params["lr"]
weight_decay_opt = best_params["weight_decay"]

x_full_opt = gera_matriz_design_tempo_continuo(dados_treino)
y_full_opt = dados_treino["pseudo"].astype(float).values.reshape(-1, 1)
full_ds = TensorDataset(
    torch.tensor(x_full_opt.values.astype(np.float32)),
    torch.tensor(y_full_opt.astype(np.float32)),
)
full_loader = DataLoader(full_ds, batch_size=batch_size_opt, shuffle=True)

model_optuna = rede_optuna(input_dim=x_full_opt.shape[1], layers=layers_opt)
criterion = nn.MSELoss()
optimizer = optim.Adam(model_optuna.parameters(), lr=lr_opt, weight_decay=weight_decay_opt)

for epoch in range(100):
    model_optuna.train()
    for xb, yb in full_loader:
        optimizer.zero_grad()
        preds = model_optuna(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

In [19]:
# Predicao para os dados de teste

dados_ids_teste = dados_teste[["id_paciente"] + covariaveis].drop_duplicates().sort_values("id_paciente")
linhas_teste_evento = []
for _, linha_individuo in dados_ids_teste.iterrows():
    for tempo_evento in tempos_avaliacao_teste:
        linha = {
            "id_paciente": int(linha_individuo["id_paciente"]),
            "tempo": float(tempo_evento),
        }
        for col in covariaveis:
            linha[col] = linha_individuo[col]
        linhas_teste_evento.append(linha)
dados_teste_evento = pd.DataFrame(linhas_teste_evento)

x_pred_C = gera_matriz_design_tempo_continuo(dados_teste_evento)
x_pred_C = x_pred_C.reindex(columns=x_full_opt.columns, fill_value=0.0)
model_optuna.eval()
with torch.no_grad():
    pred_s_C = model_optuna(torch.tensor(x_pred_C.values.astype(np.float32))).numpy().reshape(-1)

pred_event_C = dados_teste_evento[["id_paciente", "tempo"]].copy()
pred_event_C["pred_s"] = pred_s_C
pred_event_C = pred_event_C.sort_values(["id_paciente", "tempo"]).reset_index(drop=True)

dir_saida = Path("../dados/saida")
dir_saida.mkdir(parents=True, exist_ok=True)
arquivo_saida_C = dir_saida / "predicao-rede-C-optuna.csv"
pred_event_C_out = pred_event_C.rename(columns={"id_paciente": "ID", "tempo": "TIME", "pred_s": "PRED_S"})
pred_event_C_out.to_csv(arquivo_saida_C, index=False)